# 贝叶斯定理（Bayes' Theorem）

对应课程：`phases/01-math-foundations/07-bayes-theorem`

> 概率关乎你所预期的，贝叶斯定理关乎你所学到的。

本 notebook 把 `bayes.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `bayes.py`。

**贯穿全课的模式：** 先验 × 似然 / 证据 = 后验。新数据把昨天的后验变成今天的先验。


## 学习目标（Learning Objectives）

- 用贝叶斯定理从先验、似然、证据算后验
- 从零做带 Laplace 平滑、对数空间的朴素贝叶斯文本分类
- 比较 MLE 和 MAP，并说明 MAP 对应 L2 正则
- 用 Beta-Binomial 共轭先验做序列更新 / A/B 测试


## 0. 依赖


In [1]:
import math
from collections import defaultdict


## 1. 贝叶斯定理（二分类证据）

对“有病 / 没病”和“检测阳性”：

$$
P(\text{sick}\mid +)
= \frac{P(+ \mid \text{sick})\,P(\text{sick})}{P(+ \mid \text{sick})P(\text{sick})+P(+ \mid \text{healthy})P(\text{healthy})}
$$

分子是真阳性贡献，分母再加假阳性贡献。疾病越稀有，假阳性越容易淹没真阳性。


In [2]:
def bayes(prior, likelihood, false_positive_rate):
    """prior=P(H)，likelihood=P(E|H)，FPR=P(E|¬H)。返回 P(H|E)。"""
    evidence = likelihood * prior + false_positive_rate * (1 - prior)
    posterior = likelihood * prior / evidence
    return posterior


# 罕见病：1% 先验，99% 真阳性率，1% 假阳性率
prior, tpr, fpr = 0.01, 0.99, 0.01
post = bayes(prior, tpr, fpr)
print("prior =", prior)
print("P(sick | +) =", round(post, 4), "  (这里恰好是 0.5)")
print("阳性并不等于 99% 有病：先验太小，假阳性人数和真阳性一样多")


prior = 0.01
P(sick | +) = 0.5   (这里恰好是 0.5)
阳性并不等于 99% 有病：先验太小，假阳性人数和真阳性一样多


## 2. 顺序更新：阳性当作下一次的先验

独立重复检测时，把后验喂回 `bayes`。函数返回每一步后验列表，而不是只打印。

两次阳性之后，1% 先验会被抬到约 99%（在 TPR=FPR 对称的这个数字下，第一次到 50%，第二次到 99%）。


In [3]:
def sequential_bayes(prior, likelihood, false_positive_rate, num_tests):
    """重复把后验当作先验。返回 [prior, after_1, ..., after_n]。"""
    current = prior
    history = [current]
    for _ in range(num_tests):
        current = bayes(current, likelihood, false_positive_rate)
        history.append(current)
    return history


hist = sequential_bayes(0.01, 0.99, 0.01, 2)
for i, p in enumerate(hist):
    label = "prior" if i == 0 else f"after test {i}"
    print(f"{label}: {p:.6f}")


prior: 0.010000
after test 1: 0.500000
after test 2: 0.990000


## 3. 朴素贝叶斯分类器

假设词之间条件独立：

$$
P(c\mid w_1,\ldots,w_n)\propto P(c)\prod_i P(w_i\mid c)
$$

实现要点：

- 拉普拉斯平滑：$(\mathrm{count}+\alpha)/(\mathrm{total}+\alpha|V|)$，避免未见词概率为 0
- 对数空间：把乘积变成求和，防止下溢
- `predict_proba`：对各类 log-score 做稳定 softmax


In [4]:
class NaiveBayes:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        """按类统计文档数、词频、总词数，并收集词表。"""
        for doc, label in zip(documents, labels):
            self.class_counts[label] += 1
            words = doc.lower().split()
            for word in words:
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def _log_prior(self, cls):
        total_docs = sum(self.class_counts.values())
        return math.log(self.class_counts[cls] / total_docs)

    def _log_likelihood(self, word, cls):
        count = self.word_counts[cls].get(word, 0)
        total = self.class_word_totals[cls]
        vocab_size = len(self.vocab)
        return math.log(
            (count + self.smoothing) / (total + self.smoothing * vocab_size)
        )

    def predict(self, document):
        """返回 log-score 最高的类。"""
        words = document.lower().split()
        best_class = None
        best_score = float("-inf")
        for cls in self.class_counts:
            score = self._log_prior(cls)
            for word in words:
                score += self._log_likelihood(word, cls)
            if score > best_score:
                best_score = score
                best_class = cls
        return best_class

    def predict_proba(self, document):
        """log-score 减 max 再 exp，归一化成各类概率。"""
        words = document.lower().split()
        scores = {}
        for cls in self.class_counts:
            score = self._log_prior(cls)
            for word in words:
                score += self._log_likelihood(word, cls)
            scores[cls] = score
        max_score = max(scores.values())
        exp_scores = {cls: math.exp(s - max_score) for cls, s in scores.items()}
        total = sum(exp_scores.values())
        return {cls: exp_scores[cls] / total for cls in exp_scores}


docs = [
    "win free money now",
    "free lottery winner",
    "meeting tomorrow noon",
    "project update attached",
]
labels = ["spam", "spam", "ham", "ham"]

nb = NaiveBayes(smoothing=1.0)
nb.train(docs, labels)
print("vocab size =", len(nb.vocab))

for msg in ["free money winner", "project meeting tomorrow"]:
    pred = nb.predict(msg)
    proba = nb.predict_proba(msg)
    print(f"{msg!r} -> {pred}  proba={ {k: round(v, 3) for k, v in proba.items()} }")


vocab size = 12
'free money winner' -> spam  proba={'spam': 0.911, 'ham': 0.089}
'project meeting tomorrow' -> ham  proba={'spam': 0.096, 'ham': 0.904}


## 4. Beta–Binomial：共轭更新

均匀先验 $\mathrm{Beta}(1,1)$。观察到 $s$ 次成功、$f$ 次失败后：

$$
\mathrm{Beta}(\alpha,\beta)\;\xrightarrow{s,f}\;\mathrm{Beta}(\alpha+s,\;\beta+f)
$$

后验均值 $(\alpha+s)/(\alpha+\beta+s+f)$。7 成功 3 失败 → $\mathrm{Beta}(8,4)$，均值 $8/12=2/3$。


In [5]:
def beta_update(alpha, beta_param, successes, failures):
    """共轭：成功加到 alpha，失败加到 beta。"""
    return alpha + successes, beta_param + failures


a, b = beta_update(1, 1, 7, 3)
mean = a / (a + b)
var = (a * b) / ((a + b) ** 2 * (a + b + 1))
print(f"prior Beta(1,1) + 7 success / 3 fail")
print(f"posterior Beta({a}, {b})")
print(f"mean = {mean:.4f}  (期望 0.6667)")
print(f"std  = {var ** 0.5:.4f}")


prior Beta(1,1) + 7 success / 3 fail
posterior Beta(8, 4)
mean = 0.6667  (期望 0.6667)
std  = 0.1307


## 5. MLE vs MAP = L2（学习目标）

抛硬币 10 次出现 7 次正面。MLE 是 $7/10=0.7$。MAP 在 $\mathrm{Beta}(\alpha,\beta)$ 先验下是 $(h+\alpha-1)/(n+\alpha+\beta-2)$。先验越强，估计越被拉向 0.5——和 L2 把权重拉向 0 是同一件事。


In [6]:
heads, total = 7, 10
mle = heads / total
map_mild = (heads + 2 - 1) / (total + 2 + 2 - 2)
map_strong = (heads + 10 - 1) / (total + 10 + 10 - 2)
print(f"MLE = {mle:.4f}")
print(f"MAP Beta(2,2)  = {map_mild:.4f}   (轻轻拉向 0.5)")
print(f"MAP Beta(10,10)= {map_strong:.4f}  (强拉向 0.5，像更重的 L2)")


MLE = 0.7000
MAP Beta(2,2)  = 0.6667   (轻轻拉向 0.5)
MAP Beta(10,10)= 0.5714  (强拉向 0.5，像更重的 L2)


## 对照表

| 函数 | 角色 |
|------|------|
| `bayes` | 二分类：先验、TPR、FPR → 后验 |
| `sequential_bayes` | 把后验链式喂回去，返回历史 |
| `NaiveBayes.train` | 计类频、词频、词表 |
| `NaiveBayes.predict` | 对数空间 MAP 类 |
| `NaiveBayes.predict_proba` | 稳定 softmax 得到类概率 |
| `beta_update` | Beta–Binomial 共轭加计数 |

要看完整打印 demo（含 MLE/MAP、A/B 测试），运行：

```bash
python bayes.py
```
